# DSFB Phosphoric Colab Reproducibility Run

This notebook starts from a fresh Colab runtime, clones the repository, installs QEMU/OVMF dependencies, runs the active DSFB court verification gate, and displays the forensic evidence outputs.

Claim scope: a passing run means this Colab runtime, at the resolved git commit and package versions printed below, passed the checked commands. It is not a claim of bit-identical behavior across every host, QEMU, OVMF, filesystem-tool, or kernel version.

In [ ]:
import os

REPO_URL = os.environ.get("DSFB_REPO_URL", "https://github.com/infinityabundance/dsfb.git")
REF = os.environ.get("DSFB_REF", "main")
WORKDIR = os.environ.get("DSFB_WORKDIR", "/content/dsfb-repro")
PROJECT = f"{WORKDIR}/crates/dsfb-phosphoric"

os.environ.update({
    "REPO_URL": REPO_URL,
    "REF": REF,
    "WORKDIR": WORKDIR,
    "PROJECT": PROJECT,
})

print(f"REPO_URL={REPO_URL}")
print(f"REF={REF}")
print(f"WORKDIR={WORKDIR}")
print(f"PROJECT={PROJECT}")

## 1. Install host dependencies

Colab runtimes are ephemeral. This cell installs the host tools required by the QEMU/OVMF verification path.

In [ ]:
%%bash
set -euxo pipefail
sudo apt-get update
sudo DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends \
  git make coreutils dosfstools mtools qemu-system-x86 ovmf

## 2. Fresh clone

The clone directory is removed first so each notebook run starts from repository source, not from previous Colab state.

In [ ]:
%%bash
set -euxo pipefail
rm -rf "$WORKDIR"
git clone "$REPO_URL" "$WORKDIR"
cd "$WORKDIR"
git checkout "$REF"
git rev-parse HEAD | tee /content/dsfb_colab_commit.txt
test -d "$PROJECT"
find "$PROJECT" -maxdepth 1 -type f -name README.md -print

## 3. Environment evidence

This records the commit, QEMU version, OVMF files, and package versions used by this run.

In [ ]:
%%bash
set -euxo pipefail
cd "$PROJECT"
{
  echo "repo_url=$(git remote get-url origin)"
  echo "commit=$(git rev-parse HEAD)"
  echo "ref=$REF"
  echo
  echo "## qemu-system-x86_64 --version"
  qemu-system-x86_64 --version | sed -n '1,3p'
  echo
  echo "## apt package versions"
  dpkg-query -W -f='${binary:Package}\t${Version}\n' \
    git make coreutils dosfstools mtools qemu-system-x86 ovmf || true
  echo
  echo "## OVMF firmware candidates"
  find /usr/share/OVMF /usr/share/edk2 -type f -name '*.fd' -print 2>/dev/null | sort || true
} | tee /content/dsfb-environment.txt

## 4. Run the active verification gate

The pass/fail gate is `make -k verify-court-active`. Output is saved to `/content/verify-court-active.log`.

In [ ]:
%%bash
set -euxo pipefail
cd "$PROJECT"
make -k verify-court-active 2>&1 | tee /content/verify-court-active.log

## 5. Build the release bundle

The bundle is an untracked review artifact. Its contents and checksums are printed in later cells.

In [ ]:
%%bash
set -euxo pipefail
cd "$PROJECT"
tools/release/package_release.sh | tee /content/release_path.txt
release_path="$(tail -n 1 /content/release_path.txt)"
test -f "$release_path"
cp "$release_path" /content/
tar -tzf "$release_path" | tee /content/release-contents.txt

## 6. Collect evidence

This creates `/content/dsfb-evidence.txt` with the DSFB QEMU markers, manifest, verdict text, key hashes, release checksums, and release archive listing.

In [ ]:
%%bash
set -euxo pipefail
cd "$PROJECT"
release_path="$(tail -n 1 /content/release_path.txt)"
{
  echo "# DSFB Colab evidence"
  date -u '+generated_utc=%Y-%m-%dT%H:%M:%SZ'
  echo "repo_url=$(git remote get-url origin)"
  echo "commit=$(git rev-parse HEAD)"
  echo
  echo "## qemu-debug.log"
  sed -n '1,120p' build/uefi-demo/dsfb/qemu-debug.log
  echo
  echo "## linked-artifact.txt"
  sed -n '1,140p' build/uefi-demo/dsfb/linked-artifact.txt
  echo
  echo "## verdict: dsfb_demo.expect"
  cat tools/verify/fixtures/verdicts/dsfb_demo.expect
  echo
  echo "## verdict: mmio_boundary_violation.expect"
  cat tools/verify/fixtures/verdicts/mmio_boundary_violation.expect
  echo
  echo "## sha256"
  sha256sum \
    build/uefi-demo/dsfb/esp/EFI/BOOT/BOOTX64.EFI \
    build/uefi-demo/dsfb/dsfb_demo.pfi \
    tests/golden/dsfb_demo.pfi \
    tools/verify/fixtures/verdicts/dsfb_demo.expect \
    tools/verify/fixtures/verdicts/mmio_boundary_violation.expect \
    "$release_path" \
    build/release/SHA256SUMS
  echo
  echo "## release SHA256SUMS"
  cat build/release/SHA256SUMS
  echo
  echo "## release contents"
  tar -tzf "$release_path"
} | tee /content/dsfb-evidence.txt

## 7. Display evidence files

In [ ]:
from pathlib import Path
import os

project = Path(os.environ["PROJECT"])
paths = [
    Path("/content/dsfb-environment.txt"),
    Path("/content/verify-court-active.log"),
    project / "build/uefi-demo/dsfb/qemu-debug.log",
    project / "build/uefi-demo/dsfb/linked-artifact.txt",
    project / "tools/verify/fixtures/verdicts/dsfb_demo.expect",
    project / "tools/verify/fixtures/verdicts/mmio_boundary_violation.expect",
    project / "build/release/SHA256SUMS",
    Path("/content/release-contents.txt"),
    Path("/content/dsfb-evidence.txt"),
]

for path in paths:
    print(f"\n===== {path} =====")
    text = path.read_text(errors="replace")
    print(text[:20000])
    if len(text) > 20000:
        print("\n[truncated for notebook display]")

## 8. Download review artifacts

This cell is Colab-specific. It downloads the release tarball, verification log, environment record, evidence record, release contents, and release checksums.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import files
except Exception as exc:
    print(f"google.colab download helper is unavailable in this runtime: {exc}")
else:
    project = Path(os.environ["PROJECT"])
    release_path = Path("/content/release_path.txt").read_text().strip().splitlines()[-1]
    download_paths = [
        release_path,
        "/content/verify-court-active.log",
        "/content/dsfb-environment.txt",
        "/content/dsfb-evidence.txt",
        "/content/release-contents.txt",
        str(project / "build/release/SHA256SUMS"),
    ]
    for path in download_paths:
        print(f"Downloading {path}")
        files.download(path)